In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor

In [42]:
df = pd.read_csv('House_Rent_Dataset.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4746 entries, 0 to 4745
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Posted On          4746 non-null   object
 1   BHK                4746 non-null   int64 
 2   Rent               4746 non-null   int64 
 3   Size               4746 non-null   int64 
 4   Floor              4746 non-null   object
 5   Area Type          4746 non-null   object
 6   Area Locality      4746 non-null   object
 7   City               4746 non-null   object
 8   Furnishing Status  4746 non-null   object
 9   Tenant Preferred   4746 non-null   object
 10  Bathroom           4746 non-null   int64 
 11  Point of Contact   4746 non-null   object
dtypes: int64(4), object(8)
memory usage: 445.1+ KB


In [6]:
df.rename({'Posted On': 'Posted_On',
           'Area Type': 'Area_Type',
           'Area Locality': 'Area_Locality',
           'Furnishing Status': 'Furnishing_Status',
           'Tenant Preferred': 'Tenant_Preferred',
           'Point of Contact': 'Point_of_Contact'
          }, axis = 1, inplace = True)

In [16]:
df.head()

,Posted_On,BHK,Rent,Size,Floor,Area_Type,Area_Locality,City,Furnishing_Status,Tenant_Preferred,Bathroom,Point_of_Contact
0,2022-05-18,2,10000,1100,Ground out of 2,Super Area,Bandel,Kolkata,Unfurnished,Bachelors/Family,2,Contact Owner
1,2022-05-13,2,20000,800,1 out of 3,Super Area,"Phool Bagan, Kankurgachi",Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
2,2022-05-16,2,17000,1000,1 out of 3,Super Area,Salt Lake City Sector 2,Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
3,2022-07-04,2,10000,800,1 out of 2,Super Area,Dumdum Park,Kolkata,Unfurnished,Bachelors/Family,1,Contact Owner
4,2022-05-09,2,7500,850,1 out of 2,Carpet Area,South Dum Dum,Kolkata,Unfurnished,Bachelors,1,Contact Owner


In [15]:
df.Posted_On = pd.to_datetime(df.Posted_On)

In [18]:
df.describe()

,Posted_On,BHK,Rent,Size,Bathroom
count,4746,4746.000000,4.746000e+03,4746.000000,4746.000000
mean,2022-06-07 18:01:40.126422272,2.083860,3.499345e+04,967.490729,1.965866
min,2022-04-13 00:00:00,1.000000,1.200000e+03,10.000000,1.000000
25%,2022-05-20 00:00:00,2.000000,1.000000e+04,550.000000,1.000000
50%,2022-06-10 00:00:00,2.000000,1.600000e+04,850.000000,2.000000
75%,2022-06-28 00:00:00,3.000000,3.300000e+04,1200.000000,2.000000
max,2022-07-11 00:00:00,6.000000,3.500000e+06,8000.000000,10.000000
std,NaN,0.832256,7.810641e+04,634.202328,0.884532


In [43]:
cat_cols = ['Floor', 'Area_Type', 'Area_Locality', 'City' , 'Furnishing_Status', 'Tenant_Preferred', 'Point_of_Contact']

In [32]:
X = df.drop(['Posted_On', 'Rent'] , axis = 1)
y = df.Rent

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state=42)

In [35]:
param_grid = {
    'iterations': [100, 500, 1000],
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5]
}

In [46]:
model = CatBoostRegressor(verbose=0, random_seed=42, cat_features=cat_cols)
grid_search = GridSearchCV(model, param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1, verbose=2)

In [47]:
grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 81 candidates, totalling 243 fits


,estimator,<catboost.cor...0028702928830>
,param_grid,"{'depth': [4, 6, ...], 'iterations': [100, 500, ...], 'l2_leaf_reg': [1, 3, ...], 'learning_rate': [0.01, 0.05, ...]}"
,scoring,'neg_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False


In [48]:
print("Лучшие параметры:", grid_search.best_params_)

Лучшие параметры: {'depth': 6, 'iterations': 500, 'l2_leaf_reg': 1, 'learning_rate': 0.01}


In [49]:
param_dist_rand = {
    'iterations': [100, 500, 1000, 2000],
    'depth': list(range(4, 11)),
    'learning_rate': np.logspace(-2, -1, 10),
    'l2_leaf_reg': np.logspace(0, 2, 10),
    'random_strength': list(range(1, 11)),
    'bagging_temperature': list(range(0, 11))
}

In [52]:
model_rand = CatBoostRegressor(verbose=0, random_seed=42, cat_features=cat_cols)
random_search = RandomizedSearchCV(model_rand, param_dist_rand, n_iter=80, cv=3, 
                                   scoring='neg_mean_squared_error', n_jobs=-1, random_state=42, verbose=2)

In [53]:
random_search.fit(X_train, y_train)

Fitting 3 folds for each of 80 candidates, totalling 240 fits


,estimator,<catboost.cor...0028702929310>
,param_distributions,"{'bagging_temperature': [0, 1, ...], 'depth': [4, 5, ...], 'iterations': [100, 500, ...], 'l2_leaf_reg': array([ 1. ...100. ]), ...}"
,n_iter,80
,scoring,'neg_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [54]:
print("Лучшие параметры:", random_search.best_params_)

Лучшие параметры: {'random_strength': 4, 'learning_rate': np.float64(0.01), 'l2_leaf_reg': np.float64(2.7825594022071245), 'iterations': 2000, 'depth': 5, 'bagging_temperature': 6}


In [57]:
r2_grid = r2_score(grid_search.best_estimator_.predict(X_test), y_test)
r2_grid

0.516866467351787

In [58]:
r2_rand = r2_score(random_search.best_estimator_.predict(X_test), y_test)
r2_rand

0.45201838683180495